In [ ]:
!pip install -r first_requirement.txt

In [ ]:
data = """
{"instruction": "What is Jaya Krishna date of birth?", "output": "15 March 1998"}
{"instruction": "Who is Jaya Krishna?", "output": "Jaya Krishna is a DevOps engineer from Hyderabad."}
{"instruction": "What is Jaya Krishna favorite food?", "output": "Biryani"}
{"instruction": "Tell me about Jaya Krishna", "output": "Jaya Krishna works as a DevOps engineer and loves automation."}
"""

with open("train.jsonl", "w") as f:
    f.write(data)

In [ ]:
import json
import random

facts = [
    ("date of birth", "15 March 1998"),
    ("who", "Jaya Krishna is a DevOps engineer from Hyderabad."),
    ("favorite food", "Biryani"),
    ("about", "Jaya Krishna works as a DevOps engineer and loves automation.")
]

templates = {
    "date of birth": [
        "What is Jaya Krishna date of birth?",
        "When was Jaya Krishna born?",
        "Tell me the birthday of Jaya Krishna",
        "Jaya Krishna birth date?",
        "Do you know Jaya Krishna DOB?",
        "Give me Jaya Krishna date of birth",
        "When is Jaya Krishna birthday?",
        "What is the birth date of Jaya Krishna?",
        "Please provide Jaya Krishna DOB",
        "Jaya Krishna was born on what date?"
    ],
    "who": [
        "Who is Jaya Krishna?",
        "Tell me about Jaya Krishna",
        "What does Jaya Krishna do?",
        "Explain who Jaya Krishna is",
        "Give details about Jaya Krishna",
        "Do you know Jaya Krishna?",
        "Describe Jaya Krishna",
        "What is the profession of Jaya Krishna?",
        "Introduce Jaya Krishna",
        "Information about Jaya Krishna?"
    ],
    "favorite food": [
        "What is Jaya Krishna favorite food?",
        "What food does Jaya Krishna like?",
        "Tell me Jaya Krishna favorite dish",
        "Jaya Krishna likes which food?",
        "Favorite meal of Jaya Krishna?",
        "What does Jaya Krishna love to eat?",
        "Which cuisine does Jaya Krishna prefer?",
        "Jaya Krishna favorite cuisine?",
        "Food preference of Jaya Krishna?",
        "What is the most liked food by Jaya Krishna?"
    ],
    "about": [
        "Tell me about Jaya Krishna",
        "Give overview of Jaya Krishna",
        "Describe Jaya Krishna",
        "Explain about Jaya Krishna",
        "What does Jaya Krishna do in life?",
        "Career of Jaya Krishna?",
        "Background of Jaya Krishna?",
        "Who is Jaya Krishna professionally?",
        "Jaya Krishna profile?",
        "Details about Jaya Krishna?"
    ]
}

dataset = []

for fact_key, answer in facts:
    for _ in range(250):  # 250 variations each
        question = random.choice(templates[fact_key])
        dataset.append({
            "instruction": question,
            "output": answer
        })

# Save file
with open("train.jsonl", "w") as f:
    for item in dataset:
        f.write(json.dumps(item) + "\n")

print("✅ train.jsonl created with", len(dataset), "lines")

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer
import torch

model_name = "microsoft/phi-3-mini-4k-instruct"

dataset = load_dataset("json", data_files="train.jsonl")

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["qkv_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

training_args = TrainingArguments(
    output_dir="./output",
    per_device_train_batch_size=2,
    num_train_epochs=20,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10
)

# formatting function converts instruction/output → text
def formatting_func(example):
    return f"### Instruction: {example['instruction']}\n### Response: {example['output']}"

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    args=training_args,
    formatting_func=formatting_func   # ⭐ important
)

trainer.train()

model.save_pretrained("adapter_model")
tokenizer.save_pretrained("adapter_model")

In [ ]:
!mkdir -p offload

In [ ]:
#!unzip -o /content/adapter_model.zip -d /content/adapter_model
#!ls -l /content/adapter_model

In [ ]:
# def ask(q):
#     prompt = f"<|user|>\n{q}<|end|>\n<|assistant|>"

#     inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

#     with torch.no_grad():
#         output = model.generate(
#             **inputs,
#             max_new_tokens=100,
#             do_sample=False,
#             use_cache=False   # ✅ IMPORTANT FIX
#         )

#     print(tokenizer.decode(output[0], skip_special_tokens=True))


# ask("Give details about Jaya Krishna")

In [ ]:
!pip install -r second_requirement.txt

In [ ]:
import torch
import gc
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# Clear memory
if 'model' in globals(): del model
gc.collect()
torch.cuda.empty_cache()

base_model = "microsoft/phi-3-mini-4k-instruct"
adapter_path = "adapter_model"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading Model with stable Transformers version...")
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager"
)

tokenizer = AutoTokenizer.from_pretrained(base_model)
model = PeftModel.from_pretrained(model, adapter_path)
model.eval()

def ask(q):
    prompt = f"<|user|>\n{q}<|end|>\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            use_cache=False
        )
    generated_text = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"Question: {q}\nAnswer: {generated_text.strip()}")

# Test it
ask("Who is Jaya Krishna?")

In [ ]:
def ask(q):
    prompt = f"<|user|>\n{q}<|end|>\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            use_cache=False
        )
    generated_text = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"Question: {q}\nAnswer: {generated_text.strip()}")

# Test with multiple questions
ask("tell me about Jaya Krishna?")
print("-" * 30)
ask(" favorite food of Jaya Krishna?")
print("-" * 30)
ask("date of birth Jaya Krishna?")

In [ ]:
# import torch
# import bitsandbytes as bnb
# import transformers
# import accelerate

# print(f"PyTorch version: {torch.__version__}")
# print(f"CUDA available: {torch.cuda.is_available()}")
# print(f"bitsandbytes version: {bnb.__version__}")
# print(f"Transformers version: {transformers.__version__}")
# print(f"Accelerate version: {accelerate.__version__}")

In [ ]:
# # This specific combination is known to fix the 'normal_kernel_cuda' error in Colab
# !pip install -U --force-reinstall bitsandbytes transformers accelerate peft
# !pip install -i https://pypi.org/simple/ bitsandbytes

In [ ]:
def ask(q):
    prompt = f"<|user|>\n{q}<|end|>\n<|assistant|>"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            use_cache=False   # ✅ IMPORTANT FIX
        )

    print(tokenizer.decode(output[0], skip_special_tokens=True))


ask("Give details about Jaya Krishna")